# M20 · Tenseurs, axes et estimateurs pondérés — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m20_tenseurs_normalisation/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m20_tenseurs_normalisation/03_notebook.ipynb`, créée par `python tools/start_chapter.py M20`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M20.1 à M20.5, M20.8 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M20.10 | `einsum` pour deux tableaux, avec des boucles · `einsum_pair` | 🔨 | ★★ | 45 |
|  | M20.11 | Moyennes et variances le long d'axes choisis · `moments_along_axes` | 🔨 | ★ | 25 |
|  | M20.12 | L'estimateur de Horvitz-Thompson d'un total · `horvitz_thompson_total` | 🔨 | ★ | 15 |
| B | M20.13 | Prédire la forme d'une réduction ou d'un einsum, puis vérifier | 🔮 | ★ | 10 |
|  | M20.14 | Simuler un sondage : Horvitz-Thompson contre la moyenne naïve, biais et variance | 🔬 | ★★ | 20 |
|  | M20.15 | Le produit de L jacobiennes I + hJ : ses valeurs singulières extrêmes quand L grandit, à hL fixé | 🎨 | ★ | 20 |
| C | M20.16 | Chasse à l'erreur : une moyenne sur le mauvais axe, une diffusion silencieuse, un ddof oublié | 🐛 | ★ | 15 |
|  | M20.17 | Un test de propriété : réduire sur deux axes d'un coup, ou l'un après l'autre | 🛠️ | ★ | 25 |
| D | M20.18 | Défi : des probabilités d'inclusion proportionnelles à la taille, et un estimateur presque sans variance | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M20")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : des formes après réductions et remodelages, la notation d'Einstein et le coût d'une contraction, la matrice de centrage et la standardisation, un sondage de Horvitz-Thompson, un schéma d'Euler en $L$ pas, et un pont.
- Écrire les trois fonctions de `mymath/tensors.py` (`einsum_pair`, `moments_along_axes`, `horvitz_thompson_total`) et les tester.
- Prédire des formes puis les voir, comparer trois estimateurs d'un total par la simulation, dessiner le produit de $L$ pas d'Euler à horizon fixé, corriger trois erreurs classiques, écrire un test de propriété, et relever un défi : des probabilités proportionnelles à la taille.

**Rappel express.** Un tableau de forme $(n_0, \ldots, n_{k-1})$ ; en ordre C, $[i_0, i_1, i_2]$ est à la position $(i_0n_1 + i_1)\,n_2 + i_2$ ; `reshape` relit la file dans l'ordre C, `transpose` change l'ordre des axes. Réduire fait disparaître l'axe nommé ; `keepdims=True` le garde, de longueur 1. Diffusion : formes alignées à droite, longueurs égales ou 1 ; $(n,)$ contre $(n, 1)$ donne $(n, n)$. Variance : $\frac{1}{N - \text{ddof}}\sum (x - \bar x)^2$. `einsum` : une lettre absente du résultat est sommée ; coût, le produit des longueurs de toutes les lettres. $\mathbf{C} = \mathbf{I} - \frac1m\mathbf{1}\mathbf{1}^\top$, projection orthogonale de trace $m - 1$ ; la standardisation de $a\mathbf{x} + b\mathbf{1}$, $a > 0$, est celle de $\mathbf{x}$. Horvitz-Thompson : $\hat Y = \sum_{i \in S} y_i/\pi_i$, sans biais ; $\mathbb{V}(\hat Y) = \sum_i y_i^2\frac{1 - \pi_i}{\pi_i}$ en tirage de Poisson. Euler : $\mathbf{I} + h\mathbf{J}$ ; $L$ pas à horizon $T = hL$ fixé restent entre $(1 - hK)^L$ et $e^{KT}$, et $(\mathbf{I} + \frac TL\mathbf{J})^L \to e^{T\mathbf{J}}$.

## Partie 0 · Vérifier tes exercices papier (✏️ M20.1 à M20.5, 🔗 M20.8)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question (une valeur exacte se tape aussi comme une fraction, `1/8`) ;
- une **forme** comme un tuple, `(7, 2)`, ou comme un texte, `"(7, 2)"` ; une forme à un seul axe s'écrit avec sa virgule, `(9,)` ;
- une liste `[0.2, -1]` ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]`, un nombre décimal s'y écrivant avec un point ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit ;
- une formule d'`einsum` entre guillemets, `"ab,b->a"`, avec les lettres que l'énoncé impose (les espaces et les majuscules ne comptent pas) ;
- une lettre entre guillemets pour un choix, `"D"` ; `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

**Ex M20.1 — Un tableau de mesures (stations × jours × heures) : les formes après réductions et remodelages**

In [ ]:
# Ex M20.1: the values you found on paper
answer_M20_1a = ...  # a) an integer
answer_M20_1b = ...  # b) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M20_1c = ...  # c) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M20_1d = ...  # d) an integer
answer_M20_1e = ...  # e) a list of 3 integers
answer_M20_1f = ...  # f) a list of 3 integers
answer_M20_1g = ...  # g) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M20_1h = ...  # h) True or False
answer_M20_1i = ...  # i) a shape: a tuple (7, 2) or a text "(7, 2)"

for letter, answer in zip("abcdefghi", [answer_M20_1a, answer_M20_1b, answer_M20_1c, answer_M20_1d, answer_M20_1e, answer_M20_1f, answer_M20_1g, answer_M20_1h, answer_M20_1i]):
    wbm.check(f"M20.1{letter}", answer)

**Ex M20.2 — Écrire en notation d'Einstein : produit matriciel, trace, forme bilinéaire, produits de matrices empilées le long d'un axe**

In [ ]:
# Ex M20.2: the values you found on paper
answer_M20_2a = ...  # a) an einsum formula, as text: "ab,b->a"
answer_M20_2b = ...  # b) an integer
answer_M20_2c = ...  # c) an einsum formula, as text: "ab,b->a"
answer_M20_2d = ...  # d) an integer
answer_M20_2e = ...  # e) an integer
answer_M20_2f = ...  # f) an integer
answer_M20_2g = ...  # g) a matrix (list of rows)
answer_M20_2h = ...  # h) an integer
answer_M20_2i = ...  # i) an integer
answer_M20_2j = ...  # j) an integer

for letter, answer in zip("abcdefghij", [answer_M20_2a, answer_M20_2b, answer_M20_2c, answer_M20_2d, answer_M20_2e, answer_M20_2f, answer_M20_2g, answer_M20_2h, answer_M20_2i, answer_M20_2j]):
    wbm.check(f"M20.2{letter}", answer)

**Ex M20.3 — La matrice de centrage, et une standardisation qui ne voit ni translation ni dilatation**

In [ ]:
# Ex M20.3: the values you found on paper
answer_M20_3a = ...  # a) an irreducible fraction, as text
answer_M20_3b = ...  # b) an irreducible fraction, as text
answer_M20_3c = ...  # c) a list of 6 numbers
answer_M20_3d = ...  # d) an integer
answer_M20_3e = ...  # e) a number
answer_M20_3f = ...  # f) a number
answer_M20_3g = ...  # g) a number (4 decimals)
answer_M20_3h = ...  # h) a number (4 decimals)
answer_M20_3i = ...  # i) a number (4 decimals)
answer_M20_3j = ...  # j) a number (4 decimals)
answer_M20_3k = ...  # k) True or False

for letter, answer in zip("abcdefghijk", [answer_M20_3a, answer_M20_3b, answer_M20_3c, answer_M20_3d, answer_M20_3e, answer_M20_3f, answer_M20_3g, answer_M20_3h, answer_M20_3i, answer_M20_3j, answer_M20_3k]):
    wbm.check(f"M20.3{letter}", answer)

**Ex M20.4 — Un sondage à probabilités inégales : l'estimateur de Horvitz-Thompson d'un total, son espérance et sa variance**

In [ ]:
# Ex M20.4: the values you found on paper
answer_M20_4a = ...  # a) a number
answer_M20_4b = ...  # b) a number (3 decimals)
answer_M20_4c = ...  # c) a number (3 decimals)
answer_M20_4d = ...  # d) a number
answer_M20_4e = ...  # e) a number
answer_M20_4f = ...  # f) a number (3 decimals)
answer_M20_4g = ...  # g) a number (3 decimals)
answer_M20_4h = ...  # h) a number (7 decimals)
answer_M20_4i = ...  # i) a number (6 decimals)
answer_M20_4j = ...  # j) an integer (a firm numbered from 1)

for letter, answer in zip("abcdefghij", [answer_M20_4a, answer_M20_4b, answer_M20_4c, answer_M20_4d, answer_M20_4e, answer_M20_4f, answer_M20_4g, answer_M20_4h, answer_M20_4i, answer_M20_4j]):
    wbm.check(f"M20.4{letter}", answer)

**Ex M20.5 — Un schéma d'Euler en L pas : la jacobienne d'un pas, celle de L pas, et la limite (1 + λT/L)ᴸ**

In [ ]:
# Ex M20.5: the values you found on paper
answer_M20_5a = ...  # a) a number
answer_M20_5b = ...  # b) a number
answer_M20_5c = ...  # c) a number (4 decimals)
answer_M20_5d = ...  # d) a number (4 decimals)
answer_M20_5e = ...  # e) a number (4 decimals)
answer_M20_5f = ...  # f) a matrix (list of rows)
answer_M20_5g = ...  # g) a number (3 decimals)
answer_M20_5h = ...  # h) a number (3 decimals)
answer_M20_5i = ...  # i) a number
answer_M20_5j = ...  # j) a matrix (list of rows)
answer_M20_5k = ...  # k) a number

for letter, answer in zip("abcdefghijk", [answer_M20_5a, answer_M20_5b, answer_M20_5c, answer_M20_5d, answer_M20_5e, answer_M20_5f, answer_M20_5g, answer_M20_5h, answer_M20_5i, answer_M20_5j, answer_M20_5k]):
    wbm.check(f"M20.5{letter}", answer)

**Ex M20.8 — Pont : où workbookIA 20 divise par une probabilité (énoncé, sans calcul)**

In [ ]:
# Ex M20.8: the values you found on paper
answer_M20_8a = ...  # a) "A", "B", "C" or "D"
answer_M20_8b = ...  # b) "A", "B", "C" or "D"

for letter, answer in zip("ab", [answer_M20_8a, answer_M20_8b]):
    wbm.check(f"M20.8{letter}", answer)

## Partie A · Ta librairie : `einsum` par des boucles, les moments le long d'axes, Horvitz-Thompson

Fiche §2 à §5. Les trois fonctions de `mymath/tensors.py`, leurs tests, et un usage de chacune.

In [ ]:
# Tools of the notebook (parts A to D)
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg

from wbm.nbtools import StopCheck, checked, drawn, filled, run_pytest_group, same, shape_text, verdict

TEST_M20 = "tests/test_m20_tensors.py"            # the tests of mymath/tensors.py

### Ex M20.10 — `einsum` pour deux tableaux, avec des boucles · `einsum_pair` 🔨 ★★ ⏱️ 45 min
**Objectif :** programmer `einsum` pour deux tableaux par des boucles sur les indices, puis s'en servir pour une moyenne pondérée le long d'un axe, une forme bilinéaire et des produits de matrices empilées.  
**Prérequis :** fiche §3 · **mymath :** `tensors.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/tensors.py`, lis la docstring du module puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. ⚠️ Le squelette n'importe rien : ajoute `import numpy as np` en tête du fichier (et `import itertools` ou `import numbers` si tu t'en sers). Ni `np.einsum`, ni `np.tensordot`, ni `np.matmul` (ni `@`), ni `np.dot`, `np.inner` ou `np.outer` dans `einsum_pair` ; ni `np.var` ni `np.std` dans `moments_along_axes` : les tests comparent tes résultats aux leurs, et un test lit ton module. Les nombres se rendent en `float` de Python, les tableaux en tableaux NumPy de flottants ; ne modifie jamais les tableaux reçus.

Écris `einsum_pair(subscripts, a, b)` (fiche §3), en suivant sa docstring : le résultat de `np.einsum(subscripts, a, b)` pour **deux** tableaux, calculé par des boucles sur toutes les valeurs des indices. La formule s'écrit sous sa forme explicite, `"ij,jk->ik"` : une lettre par axe de `a`, une virgule, une lettre par axe de `b`, puis `->` et les lettres du résultat, **sans espace** (contrairement à `np.einsum`, `einsum_pair` refuse les espaces, comme le dit sa docstring). Une lettre commune à `a` et à `b` est un indice partagé (les deux axes ont la même longueur) ; une lettre absente du résultat est sommée ; les autres indicent le résultat, dans l'ordre écrit. Le résultat est un **nouveau** tableau de flottants (à 0 dimension pour `"i,i->"`) ; `ValueError` pour une formule mal écrite (pas de `->`, pas exactement une virgule, un caractère qui n'est pas une lettre), un nombre de lettres qui n'est pas le nombre d'axes du tableau, une lettre répétée dans un même tableau ou dans le résultat, une lettre du résultat absente des deux tableaux, ou une lettre commune de deux longueurs différentes.

**L'usage.** La cellule fournie donne des températures synthétiques `T_10`, de forme $(3, 5, 24)$ (3 stations, 5 jours, 24 heures), des poids horaires `W_10` de somme 1 (les heures de 6 h à 21 h comptent double), la matrice $\mathbf{M}$ et les vecteurs $\mathbf{x}$ et $\mathbf{y}$ (`M_10`, `X_10`, `Y_10`), et deux piles de matrices, `P_10` de forme $(6, 3, 4)$ et `Q_10` de forme $(6, 4, 2)$. Avec ta fonction :
- `daily_10` : les moyennes journalières pondérées $\sum_h T[s, d, h]\,w_h$, de forme $(3, 5)$, en **un** appel ;
- a) celle de la station 2, le jour 4 (indices NumPy ; la valeur que donne le calcul, sans l'arrondir) ;
- la forme bilinéaire $\mathbf{x}^\top\mathbf{M}\mathbf{y}$ en **deux** appels : le premier reçoit `M_10` puis `Y_10` (les axes de `M_10` notés `ij`, celui de `Y_10` noté `j`), le second reçoit `X_10` (son axe noté `i`) puis le résultat du premier. b) La formule du premier appel ; c) celle du second (deux textes) ; d) la valeur de $\mathbf{x}^\top\mathbf{M}\mathbf{y}$ (un nombre) ;
- e) vrai ou faux : `einsum_pair("sij,sjk->sik", P_10, Q_10)` coïncide avec `np.matmul(P_10, Q_10)` à $10^{-12}$ près (`np.allclose(..., rtol=0, atol=1e-12)`).

La vérification compare aussi `daily_10` à `np.einsum`. Dans ta copie : combien de multiplications ta fonction fait-elle pour `daily_10`, et pour e) ? Que calcule `"sdh,h->s"`, et pourquoi n'est-ce pas la moyenne pondérée de chaque station sur ses 5 jours ? Pourquoi les boucles Python sont-elles bien plus lentes que `np.einsum` pour les mêmes multiplications ?

In [ ]:
rng_10 = np.random.default_rng(5331)
HOURS_10 = np.linspace(0.0, 23.0, 24)                          # the hours 0, 1, ..., 23
T_10 = np.round(11 + 3 * rng_10.normal(size=(3, 1, 1)) + 1.5 * rng_10.normal(size=(1, 5, 1))
                - 4 * np.cos(2 * np.pi * (HOURS_10 - 3) / 24) + rng_10.normal(0, 0.8, size=(3, 5, 24)), 1)
W_10 = np.where((HOURS_10 >= 6) & (HOURS_10 <= 21), 2.0, 1.0) / 40    # hourly weights: 6 h to 21 h count double
M_10 = np.array([[2.0, -1.0, 0.0, 3.0],
                 [1.0, 4.0, -2.0, 0.0],
                 [0.0, 2.0, 5.0, -1.0],
                 [-3.0, 0.0, 1.0, 2.0]])
X_10 = np.array([3.0, -2.0, 1.0, 0.5])
Y_10 = np.array([2.0, 1.0, -1.0, 4.0])
P_10 = rng_10.normal(size=(6, 3, 4))                            # six 3 × 4 matrices stacked along axis 0
Q_10 = rng_10.normal(size=(6, 4, 2))                            # six 4 × 2 matrices
print("T_10 :", T_10.shape, "; somme des poids W_10 :", W_10.sum())

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M20")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M20, "test_einsum_pair_ or test_module_uses_")

**L'usage** : une moyenne pondérée le long d'un axe, une forme bilinéaire en deux contractions, des produits de matrices empilées.

In [ ]:
answer_M20_10b = ...  # b) a text: the formula of the first call (M_10, then Y_10)
answer_M20_10c = ...  # c) a text: the formula of the second call (X_10, then the result of the first)
daily_10 = answer_M20_10a = answer_M20_10d = answer_M20_10e = ...
with wbm.attempt("M20.10"):          # if you call einsum_pair here: ⏳ while it is not written
    daily_10 = ...        # the (3, 5) array of the weighted daily means, in one call of einsum_pair
    answer_M20_10a = ...  # a) a number, as computed (not rounded): that of station 2, day 4
    answer_M20_10d = ...  # d) a number: xᵀ M y, in two calls of einsum_pair
    answer_M20_10e = ...  # e) True or False: einsum_pair("sij,sjk->sik", P_10, Q_10) is np.matmul(P_10, Q_10)

In [ ]:
with checked("M20.10"):
    if filled(daily_10):
        same("M20.10", 'einsum_pair("sdh,h->sd", T_10, W_10)', daily_10, np.einsum("sdh,h->sd", T_10, W_10))
    else:
        print("⏳ Ex M20.10 : la comparaison attend ton tableau daily_10.")
for letter, answer in zip("bc", [answer_M20_10b, answer_M20_10c]):
    wbm.check(f"M20.10{letter}", answer)
for letter, answer in zip("ade", [answer_M20_10a, answer_M20_10d, answer_M20_10e]):
    wbm.check(f"M20.10{letter}", answer, computed=True)

### Ex M20.11 — Moyennes et variances le long d'axes choisis · `moments_along_axes` 🔨 ★ ⏱️ 25 min
**Objectif :** programmer la moyenne et la variance d'un tableau le long d'axes choisis, les axes réduits gardés, puis standardiser un tableau de notes matière par matière et vérifier ses invariances.  
**Prérequis :** fiche §2 et §4 ; M02c · **mymath :** `tensors.py` · **Parcours :** C, E

Écris `moments_along_axes(x, axes, ddof=0)` (fiche §2), en suivant sa docstring : le couple `(mean, var)` de la moyenne et de la variance de `x` le long des axes `axes` (un entier ou un tuple d'entiers distincts ; un axe négatif compte depuis la fin), les axes réduits **gardés** avec la longueur 1, comme avec `keepdims=True`. La variance divise la somme des carrés des écarts à la moyenne par $N - \text{ddof}$, $N$ étant le nombre de valeurs réduites ensemble. Deux tableaux de flottants de même forme ; `ValueError` pour un axe hors du tableau ou répété, ou un `ddof` qui n'est pas un entier avec $0 \le \text{ddof} < N$. Elle ne centre ni ne divise `x` : elle rend seulement les deux moments.

**L'usage : standardiser des notes, matière par matière.** La cellule fournie donne les notes sur 20 de 40 élèves dans 6 matières, `NOTES_11`, de forme $(40, 6)$ (une ligne par élève, les matières dans l'ordre de `SUBJECTS_11`), et la même classe renotée, `RENOTED_11` : les maths notées sur 100 (la colonne 0 multipliée par 5) et 2 points ajoutés à tous en anglais (la colonne 3). Avec ta fonction et la diffusion :
- `Z_11` : les notes standardisées matière par matière, avec ddof = 1, $z = \frac{x - \bar x}{s}$, où $\bar x$ et $s$ sont la moyenne et l'écart type de la colonne ;
- a) `Z_11[17, 4]`, la note standardisée de l'élève 17 en philosophie (la valeur que donne le calcul, sans l'arrondir) ;
- b) vrai ou faux : la standardisation de `RENOTED_11`, faite de même, coïncide avec `Z_11` à $10^{-12}$ près (`np.allclose(..., rtol=0, atol=1e-12)`) ;
- c) la même case qu'en a), mais standardisée avec ddof = 0 (la valeur que donne le calcul, sans l'arrondir).

La vérification compare aussi `Z_11` à un calcul direct par NumPy. Dans ta copie : pourquoi b) était-il prévisible (fiche §4) ? Quel est le rapport de c) à a), et pourquoi ne dépend-il pas de la case choisie ? Si l'on ajoutait 2 points en anglais à un **seul** élève, de combien changerait la moyenne de l'anglais ?

In [ ]:
rng_11 = np.random.default_rng(5333)
SUBJECTS_11 = ["maths", "physique", "histoire", "anglais", "philosophie", "sport"]
NOTES_11 = np.clip(np.round(2 * rng_11.normal([11.0, 9.5, 12.0, 13.0, 9.0, 14.0], [3.5, 4.0, 2.5, 3.0, 3.0, 2.0],
                                              size=(40, 6))) / 2, 0, 20)      # 40 pupils × 6 subjects, marks /20
RENOTED_11 = NOTES_11 * np.array([5.0, 1, 1, 1, 1, 1]) + np.array([0.0, 0, 0, 2, 0, 0])   # maths /100, English + 2
print("NOTES_11 :", NOTES_11.shape, "; moyennes des matières :", np.round(NOTES_11.mean(axis=0), 2).tolist())

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M20")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M20, "test_moments_along_axes_")

**L'usage** : standardiser chaque matière, puis vérifier que la renotation ne change rien.

In [ ]:
Z_11 = answer_M20_11a = answer_M20_11b = answer_M20_11c = ...
with wbm.attempt("M20.11"):          # if you call moments_along_axes here: ⏳ while it is not written
    Z_11 = ...            # the (40, 6) array of the marks standardized subject by subject, ddof = 1
    answer_M20_11a = ...  # a) a number, as computed (not rounded): Z_11[17, 4]
    answer_M20_11b = ...  # b) True or False: RENOTED_11 standardized the same way gives Z_11 (atol 1e-12)
    answer_M20_11c = ...  # c) a number, as computed: the same cell, standardized with ddof = 0

In [ ]:
with checked("M20.11"):
    if filled(Z_11):
        same("M20.11", "Z_11, comparé à un calcul direct par NumPy (ddof = 1, matière par matière)", Z_11,
             (NOTES_11 - NOTES_11.mean(axis=0)) / NOTES_11.std(axis=0, ddof=1))
    else:
        print("⏳ Ex M20.11 : la vérification attend ton tableau standardisé, Z_11.")
for letter, answer in zip("abc", [answer_M20_11a, answer_M20_11b, answer_M20_11c]):
    wbm.check(f"M20.11{letter}", answer, computed=True)

### Ex M20.12 — L'estimateur de Horvitz-Thompson d'un total · `horvitz_thompson_total` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer l'estimateur de Horvitz-Thompson, puis vérifier, en énumérant tous les échantillons, qu'il est sans biais.  
**Prérequis :** fiche §5 ; M02b, R8 · **mymath :** `tensors.py` · **Parcours :** C

Écris `horvitz_thompson_total(values, included, inclusion_probs)` (fiche §5), en suivant sa docstring : l'estimation $\hat Y = \sum_{i \in S} y_i/\pi_i$ du total, en `float` ; `included` contient des booléens (ou des 0 et des 1) ; seules les valeurs des unités **tirées** servent (les autres peuvent être inconnues, `nan`) ; `ValueError` si les trois tableaux ne sont pas à une dimension et de même longueur, si `included` contient autre chose, ou si une probabilité n'est pas dans $]0, 1]$.

**L'usage.** La cellule fournie donne les habitants de 8 communes, `Y_12`, leurs probabilités d'inclusion, `PI_12`, et un échantillon tiré indépendamment (tirage de Poisson), `INCLUDED_12`. Avec ta fonction :
- a) l'estimation du nombre total d'habitants sur cet échantillon (la valeur que donne le calcul, sans l'arrondir) ;
- b) son espérance exacte : la somme, sur les $2^8 = 256$ échantillons possibles, de l'estimation de chacun multipliée par sa probabilité $\prod_{i \in S}\pi_i\prod_{i \notin S}(1 - \pi_i)$ (la valeur que donne le calcul, sans l'arrondir ; `itertools.product([False, True], repeat=8)` les énumère).

Dans ta copie : compare a) et b) au vrai total. Calcule aussi, par la même énumération, la variance de l'estimateur, et compare-la à la formule de la fiche. Quelle commune pèse le plus dans cette variance, et pourquoi ?

In [ ]:
Y_12 = np.array([1250.0, 430.0, 2900.0, 8600.0, 610.0, 15400.0, 3800.0, 970.0])   # the inhabitants of 8 communes
PI_12 = np.array([0.12, 0.05, 0.3, 0.55, 0.08, 0.95, 0.35, 0.1])                    # their inclusion probabilities
INCLUDED_12 = np.random.default_rng(5334).random(8) < PI_12                          # a Poisson draw
print("échantillon :", np.flatnonzero(INCLUDED_12).tolist(), "; taille moyenne :", round(PI_12.sum(), 2))

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M20")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M20, "test_horvitz_thompson_total_")

**L'usage** : une estimation, puis son espérance sur les 256 échantillons possibles.

In [ ]:
answer_M20_12a = answer_M20_12b = ...
with wbm.attempt("M20.12"):          # if you call horvitz_thompson_total here: ⏳ while it is not written
    answer_M20_12a = ...  # a) a number, as computed (not rounded): the estimate of the total on INCLUDED_12
    answer_M20_12b = ...  # b) a number, as computed: its exact expectation over the 256 samples

In [ ]:
for letter, answer in zip("ab", [answer_M20_12a, answer_M20_12b]):
    wbm.check(f"M20.12{letter}", answer, computed=True)
if filled(answer_M20_12b):
    verdict("M20.12", abs(answer_M20_12b - Y_12.sum()) <= 1e-9 * Y_12.sum(),
            "l'espérance vaut le total à 1e-9 près (relatif) : l'estimateur est sans biais.",
            "l'espérance n'est pas le total : chaque échantillon compte avec sa probabilité.")

## Partie B · Prédire, simuler, dessiner

Fiche §1, §3, §5 et §6. Les formes de huit expressions, prédites puis vues ; trois estimateurs d'un total comparés par la simulation ; le produit de $L$ pas d'Euler à horizon fixé.

### Ex M20.13 — Prédire la forme d'une réduction ou d'un einsum, puis vérifier 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la forme d'un tableau après une réduction, une diffusion, un `einsum` ou un remodelage, puis la voir.  
**Prérequis :** fiche §1 à §3 · **Parcours :** C, E

La cellule fournie crée quatre tableaux : une image `IMG_13` de forme $(120, 160, 3)$ (hauteur, largeur, canaux), des températures `T_13` de forme $(6, 31, 24)$ (6 stations, 31 jours, 24 heures), un coefficient par canal, `W_13`, de forme $(3,)$, et les notes `NOTES_13` de 25 élèves dans 4 matières, de forme $(25, 4)$. Prédis, **avant d'exécuter quoi que ce soit**, la forme de chaque résultat (un tuple `(7, 2)` ou un texte `"(7, 2)"` ; `(9,)` pour un seul axe) :
- a) `IMG_13.mean(axis=(0, 1))` ;
- b) `IMG_13.sum(axis=2, keepdims=True)` ;
- c) `np.einsum("hwc,c->hw", IMG_13, W_13)` ;
- d) `T_13 - T_13.mean(axis=(1, 2), keepdims=True)` ;
- e) `np.einsum("sdh,sdh->s", T_13, T_13)` ;
- f) `T_13.reshape(6, -1)` ;
- g) `NOTES_13.mean(axis=1) - NOTES_13.mean(axis=1, keepdims=True)` ;
- h) `T_13.transpose(1, 0, 2).reshape(31, -1)`.

L'expérience calcule chaque expression et affiche sa forme.

Écris tes huit prédictions, vérifie-les, puis lance l'expérience.

In [ ]:
rng_13 = np.random.default_rng(5335)
IMG_13 = rng_13.random((120, 160, 3))                          # an image: (height, width, channels)
T_13 = np.round(rng_13.normal(12.0, 5.0, size=(6, 31, 24)), 1)  # 6 stations × 31 days × 24 hours
W_13 = np.array([2.0, -1.0, -1.0])                             # a coefficient per channel: red against the others
NOTES_13 = np.round(rng_13.uniform(4, 19, size=(25, 4)) * 2) / 2  # 25 pupils × 4 subjects

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M20_13a = ...  # a) a shape, (7, 2) or "(7, 2)", written BEFORE the experiment: IMG_13.mean(axis=(0, 1))
prediction_M20_13b = ...  # b) IMG_13.sum(axis=2, keepdims=True)
prediction_M20_13c = ...  # c) np.einsum("hwc,c->hw", IMG_13, W_13)
prediction_M20_13d = ...  # d) T_13 - T_13.mean(axis=(1, 2), keepdims=True)
prediction_M20_13e = ...  # e) np.einsum("sdh,sdh->s", T_13, T_13)
prediction_M20_13f = ...  # f) T_13.reshape(6, -1)
prediction_M20_13g = ...  # g) NOTES_13.mean(axis=1) - NOTES_13.mean(axis=1, keepdims=True)
prediction_M20_13h = ...  # h) T_13.transpose(1, 0, 2).reshape(31, -1)

In [ ]:
for letter, prediction in zip("abcdefgh", [prediction_M20_13a, prediction_M20_13b, prediction_M20_13c,
                                             prediction_M20_13d, prediction_M20_13e, prediction_M20_13f,
                                             prediction_M20_13g, prediction_M20_13h]):
    wbm.check(f"M20.13{letter}", shape_text(prediction))     # a tuple (25,) is shown as (25,), not as 25

**L'expérience.** Elle ne s'exécute qu'une fois tes huit prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M20_13a, prediction_M20_13b, prediction_M20_13c, prediction_M20_13d, prediction_M20_13e, prediction_M20_13f, prediction_M20_13g, prediction_M20_13h]):
    print("⏳ Écris d'abord tes huit prédictions, puis exécute à nouveau.")
else:
    with checked("M20.13"):
        expressions_13 = [("a", "IMG_13.mean(axis=(0, 1))", lambda: IMG_13.mean(axis=(0, 1))),
                          ("b", "IMG_13.sum(axis=2, keepdims=True)", lambda: IMG_13.sum(axis=2, keepdims=True)),
                          ("c", 'np.einsum("hwc,c->hw", IMG_13, W_13)', lambda: np.einsum("hwc,c->hw", IMG_13, W_13)),
                          ("d", "T_13 - T_13.mean(axis=(1, 2), keepdims=True)",
                           lambda: T_13 - T_13.mean(axis=(1, 2), keepdims=True)),
                          ("e", 'np.einsum("sdh,sdh->s", T_13, T_13)', lambda: np.einsum("sdh,sdh->s", T_13, T_13)),
                          ("f", "T_13.reshape(6, -1)", lambda: T_13.reshape(6, -1)),
                          ("g", "NOTES_13.mean(axis=1) - NOTES_13.mean(axis=1, keepdims=True)",
                           lambda: NOTES_13.mean(axis=1) - NOTES_13.mean(axis=1, keepdims=True)),
                          ("h", "T_13.transpose(1, 0, 2).reshape(31, -1)", lambda: T_13.transpose(1, 0, 2).reshape(31, -1))]
        for letter_13, text_13, compute_13 in expressions_13:
            print(f"{letter_13}) {text_13:62} forme {compute_13().shape}")

**Après l'expérience**, dans ta copie : que contient chaque résultat, en mots ? Pour g), qu'a-t-on probablement voulu calculer, et comment l'écrire ? Pour h), que contient chaque ligne, et que contiendrait la première ligne de `T_13.reshape(31, -1)`, qui a la même forme ?

### Ex M20.14 — Simuler un sondage : Horvitz-Thompson contre la moyenne naïve, biais et variance 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer par la simulation le biais et l'écart type de trois estimateurs d'un total, et voir ce que des probabilités proportionnelles à une taille bien choisie apportent.  
**Prérequis :** fiche §5 ; Ex M20.12 · **Parcours :** C

Une population de $N = 400$ entreprises (cellule fournie) : leur effectif $x_i$ et leur âge, connus (`X_14`, très dispersé : quelques grandes, beaucoup de petites ; `AGE_14`, de 2 à 30 ans, sans lien avec leur taille), et leur chiffre d'affaires $y_i$ en millions d'euros, à peu près proportionnel à l'effectif, que l'on veut estimer sans le connaître (`Y_14` ; le vrai total est `TOTAL_14`). Deux plans de sondage, chacun en tirage de Poisson, à probabilités inégales :
- `PI_14` : des probabilités proportionnelles à l'effectif, de somme 40, plafonnées à 1 (une entreprise atteint le plafond : la somme est un peu sous 40) ;
- `PI_AGE_14` : des probabilités proportionnelles à l'âge, de même somme.

La cellule fournie tire 3 000 échantillons avec chacun : `DRAWS_14` et `DRAWS_AGE_14`, deux tableaux de booléens de forme $(3\,000, 400)$, un échantillon par ligne. Calcule trois tableaux de 3 000 estimations du total, une par ligne, dans l'ordre des lignes :
- `ht_14` : Horvitz-Thompson avec `PI_14`, sur `DRAWS_14`, par ta `horvitz_thompson_total` ;
- `naive_14` : l'estimateur naïf $N\,\bar y_S$ (400 fois la moyenne des $y_i$ tirés), sur les mêmes échantillons `DRAWS_14` ;
- `ht_age_14` : Horvitz-Thompson avec `PI_AGE_14`, sur `DRAWS_AGE_14`.

Pour un estimateur, le **biais relatif** est $\frac{m}{Y} - 1$ et l'**écart type relatif** $\frac{s}{Y}$, où $m$ et $s$ sont la moyenne et l'écart type (ddof = 1) de ses 3 000 estimations ; l'**erreur type** de Monte-Carlo du biais relatif est $\frac{s}{Y\sqrt{3\,000}}$. La cellule de vérification vérifie d'abord tes réponses, compare tes tableaux aux estimations attendues et trace les trois histogrammes, avec le vrai total ; une fois tes quatre réponses justes, elle affiche ces trois nombres pour chaque estimateur.

Réponds, en pourcentages (la valeur que donne le calcul, sans l'arrondir) :
- a) l'écart type relatif de `ht_14` ;
- b) celui de `ht_age_14` ;
- c) le biais relatif de `naive_14` ;
- d) vrai ou faux : le biais relatif de `ht_14` est, en valeur absolue, inférieur à 3 erreurs types.

In [ ]:
rng_14 = np.random.default_rng(5336)
N_14 = 400                                                         # the firms
X_14 = np.round(np.exp(rng_14.normal(np.log(25), 1.2, size=N_14))) + 1      # their headcount: known
Y_14 = np.round(0.12 * X_14 * np.exp(rng_14.normal(0, 0.3, size=N_14)), 2)  # their turnover (M€): unknown
AGE_14 = np.round(rng_14.uniform(2, 30, size=N_14))                # their age, in years: known, unrelated to y
TOTAL_14 = Y_14.sum()                                              # the true total, to judge the estimators
PI_14 = np.minimum(1.0, 40 * X_14 / X_14.sum())                    # π ∝ headcount, of sum 40, capped at 1
PI_AGE_14 = PI_14.sum() * AGE_14 / AGE_14.sum()                     # π ∝ age, of the same sum
DRAWS_14 = rng_14.random((3000, N_14)) < PI_14                      # 3 000 Poisson draws with PI_14, one per row
DRAWS_AGE_14 = rng_14.random((3000, N_14)) < PI_AGE_14              # 3 000 Poisson draws with PI_AGE_14
assert DRAWS_14.sum(axis=1).min() > 0
print("Σπ =", round(PI_14.sum(), 3), "; π = 1 pour", int(np.sum(PI_14 == 1)), "entreprise(s) ; π selon l'âge : de",
      round(PI_AGE_14.min(), 4), "à", round(PI_AGE_14.max(), 4), "; vrai total :", round(TOTAL_14, 2), "M€")

In [ ]:
ht_14 = naive_14 = ht_age_14 = ...
answer_M20_14a = answer_M20_14b = answer_M20_14c = answer_M20_14d = ...
with wbm.attempt("M20.14"):          # horvitz_thompson_total: ⏳ while it is not written
    ht_14 = ...           # the 3 000 Horvitz-Thompson estimates with PI_14, one per row of DRAWS_14
    naive_14 = ...        # the 3 000 naive estimates N ȳ_S on the same rows
    ht_age_14 = ...       # the 3 000 Horvitz-Thompson estimates with PI_AGE_14, one per row of DRAWS_AGE_14
    answer_M20_14a = ...  # a) a number, as computed (not rounded): the relative standard deviation of ht_14, in %
    answer_M20_14b = ...  # b) a number, as computed: that of ht_age_14, in %
    answer_M20_14c = ...  # c) a number, as computed: the relative bias of naive_14, in %
    answer_M20_14d = ...  # d) True or False: |relative bias of ht_14| < 3 standard errors

In [ ]:
right_14 = [wbm.check(f"M20.14{letter}", answer, computed=True)
            for letter, answer in zip("abcd", [answer_M20_14a, answer_M20_14b, answer_M20_14c, answer_M20_14d])]
show_table_14 = all(right_14)

with checked("M20.14"):
    if not filled(ht_14, naive_14, ht_age_14):
        print("⏳ Ex M20.14 : le tableau et la figure attendent tes trois tableaux, ht_14, naive_14 et ht_age_14.")
        raise StopCheck
    want_14 = {"ht_14": DRAWS_14 @ (Y_14 / PI_14),
               "naive_14": N_14 * (DRAWS_14 @ Y_14) / DRAWS_14.sum(axis=1),
               "ht_age_14": DRAWS_AGE_14 @ (Y_14 / PI_AGE_14)}
    ok_14 = [same("M20.14", f"{name} (une estimation par ligne de {rows})", got, want_14[name], rtol=1e-9, atol=1e-9)
             for name, rows, got in (("ht_14", "DRAWS_14", ht_14), ("naive_14", "DRAWS_14", naive_14),
                                     ("ht_age_14", "DRAWS_AGE_14", ht_age_14))]
    if not all(ok_14):
        raise StopCheck
    rows_14 = []
    for name_f14, values_f14 in (("HT, π ∝ effectif", ht_14), ("naïf N ȳ_S", naive_14), ("HT, π ∝ âge", ht_age_14)):
        values_f14 = np.asarray(values_f14, dtype=float)
        spread_f14 = values_f14.std(ddof=1) / TOTAL_14
        rows_14.append((name_f14, 100 * (values_f14.mean() / TOTAL_14 - 1), 100 * spread_f14,
                        100 * spread_f14 / math.sqrt(len(values_f14))))
    if show_table_14:                                  # only once the four answers are right: it shows them
        print(f"{'estimateur':18} {'biais relatif':>14} {'écart type relatif':>19} {'erreur type':>12}")
        for name_f14, bias_f14, spread_f14, error_f14 in rows_14:
            print(f"{name_f14:18} {bias_f14:13.2f} % {spread_f14:17.2f} % {error_f14:10.2f} %")
    else:
        print("ℹ️ Le tableau des biais et des écarts types relatifs s'affichera quand tes quatre réponses seront justes.")
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8), sharex=True)
    try:
        for ax, (name_f14, values_f14), color_f14 in zip(axes, (("HT, π ∝ effectif", ht_14), ("naïf N ȳ_S, π ∝ effectif",
                                                                                              naive_14),
                                                                ("HT, π ∝ âge", ht_age_14)), ("C0", "C3", "C2")):
            ax.hist(np.asarray(values_f14, dtype=float), bins=60, color=color_f14, alpha=0.75)
            ax.axvline(TOTAL_14, color="k", ls="--", lw=1.2, label="vrai total")
            ax.set_title(name_f14, fontsize=10)
            ax.set_xlabel("estimation du total (M€)")
            ax.legend(fontsize=8)
        axes[0].set_ylabel("nombre d'échantillons")
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

**Après la figure**, dans ta copie : pourquoi l'estimateur naïf surestime-t-il autant (compare l'effectif moyen des entreprises tirées à celui de la population) ? Les deux estimateurs de Horvitz-Thompson sont sans biais : pourquoi `ht_14` est-il deux fois plus précis que `ht_age_14` ? Que vaut à peu près $y_i/\pi_i$ avec $\pi_i \propto x_i$, et avec $\pi_i$ proportionnel à l'âge (une jeune et grande entreprise, si elle est tirée) ? D'où vient l'écart type qui reste pour `ht_14` (pense au nombre d'entreprises tirées, qui change d'un échantillon à l'autre) ? Le défi M20.18 le fait disparaître.

### Ex M20.15 — Le produit de L jacobiennes I + hJ : ses valeurs singulières extrêmes quand L grandit, à hL fixé 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner les valeurs singulières de $(\mathbf{I} + \frac TL\mathbf{J})^L$ quand $L$ grandit à horizon fixé, leur limite $e^{T\mathbf{J}}$ et les deux bornes de la fiche.  
**Prérequis :** fiche §6 ; M12d, M14 · **Parcours :** C, E

Un pas d'Euler pour $\mathbf{x}' = \mathbf{J}\mathbf{x}$ multiplie par $\mathbf{I} + h\mathbf{J}$ ; $L$ pas de durée $h = T/L$, à horizon $T$ fixé, multiplient par $(\mathbf{I} + \frac TL\mathbf{J})^L$, qui tend vers $e^{T\mathbf{J}}$ quand $L$ grandit (fiche §6). La cellule fournie donne $\mathbf{J} = \begin{pmatrix} -0{,}3 & -1{,}7 \\ 0{,}95 & -0{,}35 \end{pmatrix}$ (`J_15` : une rotation et une contraction), l'horizon $T = 2$ (`T_15`) et les onze nombres de pas $L = 1, 2, 4, \ldots, 1\,024$ (`LS_15`).

Écris `draw_products(ax, J, T, Ls)`, qui dessine dans l'axe `ax` (sans en ouvrir d'autre) :
- les deux valeurs singulières de $(\mathbf{I} + \frac TL\mathbf{J})^L$ en fonction de $L$, chacune par **une** ligne avec des points, une valeur par $L$ de `Ls`, dans l'ordre (`np.linalg.matrix_power`, puis `np.linalg.svd(..., compute_uv=False)`, qui les rend en ordre décroissant) ;
- les deux valeurs singulières de $e^{T\mathbf{J}}$ (`scipy.linalg.expm`), par deux droites horizontales (`ax.axhline`) ;
- en pointillés (`ls="--"`), les bornes de la fiche, avec $K = \lVert\mathbf{J}\rVert_2$ (`np.linalg.norm(J, 2)`) : la droite horizontale $e^{KT}$, et $(1 - hK)^L$ aux seuls $L$ de `Ls` pour lesquels $hK < 1$ (une ligne) ;
- $L$ en échelle logarithmique (`ax.set_xscale("log", base=2)`) ; une échelle logarithmique en ordonnée aussi, une légende et un titre.

Réponds :
- a) les deux valeurs singulières de $e^{T\mathbf{J}}$, en ordre décroissant (une liste, 4 décimales) ;
- b) la plus grande valeur singulière du produit pour $L = 1\,024$ (4 décimales) ;
- c) la borne $(1 - hK)^L$ pour $L = 16$, avec $h = T/L$ (4 décimales).

La vérification appelle ta fonction et regarde le dessin, dans cet ordre, en s'arrêtant au premier ❌ : les deux courbes, les deux droites de $e^{T\mathbf{J}}$, les deux bornes en pointillés (la borne basse aux seuls $L$ où $hK < 1$) et l'échelle logarithmique en $L$.

In [ ]:
J_15 = np.array([[-0.3, -1.7],
                 [0.95, -0.35]])                # a rotation and a contraction
T_15 = 2.0                                      # the horizon
LS_15 = [2 ** k for k in range(11)]             # L = 1, 2, 4, ..., 1024

In [ ]:
def draw_products(ax, J, T, Ls):
    """In ax (and no other axes): the two singular values of (I + (T/L) J)^L against L (one line each, one point per
    L of Ls), those of expm(T J) as horizontal lines, the bounds e^(KT) and (1 − hK)^L (where hK < 1) dashed,
    K = ‖J‖₂ and h = T/L; L on a log scale."""
    raise NotImplementedError("draw_products n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M20_15a = ...  # a) a list of 2 numbers (4 decimals), decreasing: the singular values of expm(T_15 * J_15)
answer_M20_15b = ...  # b) a number (4 decimals): the largest singular value of the product for L = 1024
answer_M20_15c = ...  # c) a number (4 decimals): the bound (1 − hK)^L for L = 16

In [ ]:
for letter, answer in zip("abc", [answer_M20_15a, answer_M20_15b, answer_M20_15c]):
    wbm.check(f"M20.15{letter}", answer)
if filled(answer_M20_15a, answer_M20_15b):
    verdict("M20.15", abs(answer_M20_15b - answer_M20_15a[0]) < 1e-2,
            "pour L = 1 024, la plus grande valeur singulière du produit est à moins de 1e-2 de celle de exp(TJ).",
            "pour L = 1 024, le produit devrait être à moins de 1e-2 de exp(TJ) : relis a) et b).")

with checked("M20.15"):
    def report_15(ok, success, failure):
        """One ✅ line, or one ❌ line that stops the check there (the figure is still shown)."""
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok:
            raise StopCheck

    K_c15 = np.linalg.norm(J_15, 2)
    values_c15 = np.array([np.linalg.svd(np.linalg.matrix_power(np.eye(2) + (T_15 / L) * J_15, L), compute_uv=False)
                           for L in LS_15])
    limit_c15 = np.linalg.svd(scipy.linalg.expm(T_15 * J_15), compute_uv=False)
    lower_c15 = [(L, (1 - T_15 / L * K_c15) ** L) for L in LS_15 if T_15 / L * K_c15 < 1]
    fig_15, ax_15 = plt.subplots(figsize=(7.5, 4.8))
    try:
        opened_15 = drawn(draw_products, ax_15, J_15.copy(), T_15, list(LS_15))
        lines_15 = [(np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float),
                     line.get_linestyle()) for line in ax_15.lines]

        def line_through_15(points, dashed=False):
            """The (xs, ys) of the first line of ax that passes by all the points (x, y), dashed or dotted when
            asked; None when there is none."""
            for xs, ys, style in lines_15:
                if dashed and style not in ("--", ":", "-."):
                    continue
                if xs.shape == ys.shape and all(np.any(np.isclose(xs, x, rtol=1e-9) & np.isclose(ys, y, rtol=1e-6))
                                                for x, y in points):
                    return xs, ys
            return None

        def level_15(value, dashed=False):
            """True when one line of ax is horizontal at value (ax.axhline, or a constant line)."""
            return any(ys.size >= 2 and np.all(np.isclose(ys[np.isfinite(ys)], value, rtol=1e-6))
                       and (not dashed or style in ("--", ":", "-.")) for _, ys, style in lines_15)

        report_15(not opened_15, "draw_products dessine dans l'axe reçu",
                  "draw_products doit dessiner dans l'axe reçu, sans ouvrir de figure (ni plt.figure ni plt.subplots)")
        report_15(line_through_15(list(zip(LS_15, values_c15[:, 0]))) is not None
                  and line_through_15(list(zip(LS_15, values_c15[:, 1]))) is not None,
                  "les deux valeurs singulières du produit sont tracées en fonction de L",
                  "il manque une courbe des valeurs singulières du produit (I + (T/L) J)^L : une ligne par valeur "
                  "singulière, une valeur par L de Ls")
        report_15(level_15(limit_c15[0]) and level_15(limit_c15[1]),
                  "les deux valeurs singulières de exp(TJ) sont des droites horizontales",
                  "il manque une droite horizontale aux valeurs singulières de scipy.linalg.expm(T * J) (ax.axhline)")
        report_15(level_15(math.exp(K_c15 * T_15), dashed=True), "la borne e^(KT) est en pointillés",
                  "il manque la borne e^(KT), K = np.linalg.norm(J, 2), en pointillés (ls=\"--\")")
        lower_line_15 = line_through_15(lower_c15, dashed=True)
        report_15(lower_line_15 is not None, "la borne (1 − hK)^L est en pointillés, aux L où hK < 1",
                  "il manque la borne (1 − hK)^L, h = T/L, en pointillés, aux L de Ls pour lesquels hK < 1")
        outside_15 = sorted({int(x) if float(x).is_integer() else float(x) for x, y in zip(*lower_line_15)
                             if np.isfinite(y) and (x <= 0 or T_15 / x * K_c15 >= 1)})
        report_15(not outside_15, "la borne (1 − hK)^L n'est tracée qu'aux L où hK < 1",
                  f"la borne (1 − hK)^L est aussi tracée pour L = {outside_15}, où hK ≥ 1 : la fiche ne la donne que "
                  "pour hK < 1, ne la trace qu'à ces L")
        report_15(ax_15.get_xscale() == "log", "L est en échelle logarithmique",
                  "mets L en échelle logarithmique : ax.set_xscale(\"log\", base=2)")
    except StopCheck:
        plt.show()
        raise
    except BaseException:
        plt.close(fig_15)
        raise
    plt.show()
    verdict("M20.15", True, "les deux courbes, leurs limites et les deux bornes sont dessinées.", "")

**Pour aller plus loin** (facultatif). Le même dessin pour un champ non linéaire, l'oscillateur amorti $F(x, v) = (v,\ -x - x^3 - 0{,}3\,v)$ depuis $(1{,}2 ;\ 0)$, à horizon $T = 4$ : pour chaque $L$, le produit des jacobiennes $\mathbf{I} + h\mathbf{J}_F(\mathbf{x}_k)$ le long des points d'Euler $\mathbf{x}_0, \ldots, \mathbf{x}_{L-1}$, celle du dernier pas à gauche (fiche §6). Le notebook de solutions le trace.

**Dans ta copie** : pourquoi les deux courbes se stabilisent-elles quand $L$ grandit, alors que le nombre de facteurs double d'un point au suivant ? Pourquoi les valeurs singulières dépassent-elles 1 pour $L = 1$ et $L = 2$ (calcule $\lVert\mathbf{I} + 2\mathbf{J}\rVert$), et pourquoi les bornes sont-elles si loin des courbes ?

## Partie C · Déboguer, tester

Fiche §2. Trois erreurs classiques des réductions le long d'axes ; un test de propriété pour les moments le long de deux axes. La vérification de M20.17 s'appuie sur la référence (`ref_tensors`), pour ne pas dépendre de ton code.

In [ ]:
ref_tensors = wbm.load_mymath("ref", alias="mymath_ref").tensors   # the reference: the check of M20.17

### Ex M20.16 — Chasse à l'erreur : une moyenne sur le mauvais axe, une diffusion silencieuse, un ddof oublié 🐛 ★ ⏱️ 15 min
**Objectif :** reconnaître trois erreurs classiques des réductions le long d'axes qui ne lèvent aucune erreur Python et passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §2 · **Parcours :** C, E

Un collègue a écrit trois fonctions (cellule fournie) ; chacune a l'erreur que nomme le titre, aucune ne lève d'erreur Python, et chacune a passé la vérification qu'il a faite :
1. `subject_means_16(notes)` doit rendre la moyenne de chaque **matière** d'un tableau (élèves, matières). Il l'a vérifiée sur un tableau de 5 élèves et 5 matières : le résultat a bien la forme $(5,)$.
2. `noon_gap_16(T)` reçoit une journée de relevés, `T` de forme (stations, 24), et doit rendre l'écart moyen, sur les stations, entre le relevé de midi (l'heure 12) de chaque station et sa moyenne journalière, en valeur absolue : $\frac1n\sum_s \lvert T[s, 12] - \bar T_s\rvert$. Il l'a vérifiée sur quatre stations au même profil : la valeur attendue.
3. `sample_sd_16(x)` doit rendre l'écart type d'un échantillon de mesures répétées (l'incertitude d'une mesure), la racine de sa variance sans biais (ddof = 1). Il l'a vérifiée sur 100 000 tirages d'écart type 2 : à peu près 2.

À faire : écris `subject_means_16_fixed`, `noon_gap_16_fixed` et `sample_sd_16_fixed`, puis, avec elles (la valeur que donne le calcul, sans l'arrondir) :
- a) les moyennes des 3 matières du tableau `NOTES_16` (6 élèves, 3 matières ; une liste de 3 nombres) ;
- b) l'écart moyen de midi des 5 stations de `T_16` ;
- c) l'écart type des 5 mesures répétées d'une même longueur de `SMALL_16`, en mm.

La vérification essaie tes corrections sur des cas que tu ne vois pas : des tableaux de formes variées, des stations aux profils différents, de petits échantillons.

In [ ]:
NOTES_16 = np.array([[12.0, 8.5, 15.0],
                     [9.5, 11.0, 13.0],
                     [14.0, 10.0, 16.5],
                     [7.0, 12.5, 11.0],
                     [15.5, 9.0, 14.0],
                     [10.0, 13.5, 12.0]])            # 6 pupils × 3 subjects
HOURS_16 = np.linspace(0.0, 23.0, 24)
rng_16 = np.random.default_rng(5332)
T_16 = np.round(np.array([[8.0], [12.5], [5.0], [15.0], [10.0]])
                + np.array([[3.0], [6.0], [2.0], [8.0], [4.5]]) * np.cos(2 * np.pi * (HOURS_16 - [[14], [15], [13], [16], [14]]) / 24)
                + rng_16.normal(0, 0.5, size=(5, 24)), 1)    # one day of 5 stations, hour by hour
SMALL_16 = np.array([12.5, 14.0, 9.5, 11.0, 13.0])     # five repeated measurements of one length, in mm


# The colleague's code
def subject_means_16(notes):
    """The mean of each subject of a table (pupils, subjects): one value per subject."""
    return notes.mean(axis=1)


def noon_gap_16(T):
    """T of shape (stations, 24): one day, hour by hour. The mean, over the stations, of |T[s, 12] − daily mean of s|."""
    return float(np.mean(np.abs(T[:, 12] - T.mean(axis=1, keepdims=True))))


def sample_sd_16(x):
    """The standard deviation of a sample: the square root of its unbiased variance (ddof = 1)."""
    return float(np.std(x))


# The colleague's checks: all pass
square_16 = np.round(np.random.default_rng(5332).uniform(5, 18, size=(5, 5)), 1)
print("(1) forme des moyennes d'un tableau de 5 élèves et 5 matières :", subject_means_16(square_16).shape,
      "(attendu (5,))")
profile_16 = 10 + 4 * np.cos(2 * np.pi * (HOURS_16 - 14) / 24)
print("(2) quatre stations au même profil :", round(noon_gap_16(np.tile(profile_16, (4, 1))), 6), "; attendu",
      round(abs(profile_16[12] - profile_16.mean()), 6))
print("(3) 100 000 tirages d'écart type 2 :", round(sample_sd_16(np.random.default_rng(5332).normal(15, 2, 100_000)), 2))

In [ ]:
def subject_means_16_fixed(notes):
    """The mean of each subject of a table (pupils, subjects): one value per subject."""
    raise NotImplementedError("subject_means_16_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def noon_gap_16_fixed(T):
    """T of shape (stations, 24): one day, hour by hour. The mean, over the stations, of |T[s, 12] − daily mean of s|."""
    raise NotImplementedError("noon_gap_16_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def sample_sd_16_fixed(x):
    """The standard deviation of a sample: the square root of its unbiased variance (ddof = 1)."""
    raise NotImplementedError("sample_sd_16_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M20_16a = answer_M20_16b = answer_M20_16c = ...
with wbm.attempt("M20.16"):          # your three functions: ⏳ while they are not written
    answer_M20_16a = ...  # a) a list of 3 numbers, as computed (not rounded): the means of the subjects of NOTES_16
    answer_M20_16b = ...  # b) a number, as computed: the mean noon gap of T_16
    answer_M20_16c = ...  # c) a number, as computed: the standard deviation of SMALL_16

In [ ]:
for letter, answer in zip("abc", [answer_M20_16a, answer_M20_16b, answer_M20_16c]):
    wbm.check(f"M20.16{letter}", answer, computed=True)

results_16, parts_16 = [], []


def report_16(ok, success, failure):
    results_16.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_16(part, name, function, *args):
    """function(*args); an error of the learner's code becomes one ❌ line that names the function (the check of this
    part stops there, the final verdict stays)."""
    try:
        return function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_16(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        parts_16.append(part)
        raise StopCheck from None


def close_16(got, want):
    """True when got is a number (or an array of numbers) of the shape of want, equal to it up to rounding."""
    try:
        got = np.asarray(got, dtype=float)
    except (TypeError, ValueError):
        return False
    return got.shape == np.shape(want) and bool(np.allclose(got, want, rtol=1e-9, atol=1e-12))


with checked("M20.16"):                                       # (1) subject_means_16_fixed
    rng_c16 = np.random.default_rng(5332)
    wrong_16 = []
    for shape_c16 in ((7, 3), (4, 6), (5, 5), (12, 2)):
        notes_c16 = np.round(rng_c16.uniform(2, 19, size=shape_c16), 1)
        got_16 = call_16(1, "subject_means_16_fixed", subject_means_16_fixed, notes_c16.copy())
        want_16 = np.array([np.mean(notes_c16[:, j]) for j in range(shape_c16[1])])
        if got_16 is None or not close_16(got_16, want_16):
            wrong_16.append(f"un tableau {shape_c16[0]} × {shape_c16[1]} : attendu {np.round(want_16, 3).tolist()}, "
                            f"reçu {got_16!r}"[:220])
    report_16(not wrong_16, "(1) subject_means_16_fixed : quatre tableaux de formes variées",
              "(1) subject_means_16_fixed : " + " ; ".join(wrong_16[:2]))
    parts_16.append(1)

with checked("M20.16"):                                       # (2) noon_gap_16_fixed
    wrong_16 = []
    for n_c16 in (2, 3, 7):
        T_c16 = np.round(rng_c16.uniform(0, 20, size=(n_c16, 1)) + rng_c16.uniform(1, 8, size=(n_c16, 1))
                         * np.cos(2 * np.pi * (HOURS_16 - rng_c16.uniform(10, 18, size=(n_c16, 1))) / 24)
                         + rng_c16.normal(0, 1, size=(n_c16, 24)), 1)
        want_16 = float(np.mean([abs(T_c16[s, 12] - np.mean(T_c16[s])) for s in range(n_c16)]))
        got_16 = call_16(2, "noon_gap_16_fixed", noon_gap_16_fixed, T_c16.copy())
        if got_16 is None or not close_16(got_16, want_16):
            wrong_16.append(f"{n_c16} stations aux profils différents : attendu {want_16:.6g}, reçu {got_16!r}"[:200])
    report_16(not wrong_16, "(2) noon_gap_16_fixed : des stations aux profils différents",
              "(2) noon_gap_16_fixed : " + " ; ".join(wrong_16[:2]))
    parts_16.append(2)

with checked("M20.16"):                                       # (3) sample_sd_16_fixed
    wrong_16 = []
    for size_c16 in (2, 3, 5, 9, 40):
        x_c16 = np.round(rng_c16.normal(10, 3, size=size_c16), 2)
        want_16 = math.sqrt(sum((v - x_c16.mean()) ** 2 for v in x_c16) / (size_c16 - 1))
        got_16 = call_16(3, "sample_sd_16_fixed", sample_sd_16_fixed, x_c16.copy())
        if got_16 is None or not close_16(got_16, want_16):
            wrong_16.append(f"{size_c16} valeurs : attendu {want_16:.6g}, reçu {got_16!r}"[:160])
    report_16(not wrong_16, "(3) sample_sd_16_fixed : des échantillons de 2 à 40 valeurs",
              "(3) sample_sd_16_fixed : " + " ; ".join(wrong_16[:2]))
    parts_16.append(3)

if len(set(parts_16)) == 3:
    verdict("M20.16", all(results_16), "les trois fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pourquoi la vérification (1) ne pouvait-elle pas voir l'erreur, et quelle vérification l'aurait vue ? Quelle est la forme de `T[:, 12] - T.mean(axis=1, keepdims=True)` pour 5 stations, et que contient sa case $(i, j)$ ? Pourquoi quatre stations au même profil cachent-elles l'erreur (2) ? Pour (3), de quel facteur le résultat du collègue s'écarte-t-il sur $n$ valeurs, et pourquoi ne le voit-on pas sur 100 000 ?

### Ex M20.17 — Un test de propriété : réduire sur deux axes d'un coup, ou l'un après l'autre 🛠️ ★ ⏱️ 25 min
**Objectif :** écrire un test de propriété pour les moments le long de deux axes, et s'assurer qu'il attrape trois fonctions fausses.  
**Prérequis :** fiche §2 ; Ex M20.11 · **Parcours :** C

Un **test de propriété** vérifie, sur beaucoup de cas tirés au hasard, une loi que tout résultat juste respecte, sans connaître la réponse de chaque cas (M17, M18). La loi, ici (fiche §2) : pour un tableau `x` et deux axes distincts $a$ et $b$,
- réduire sur $a$ et $b$ d'un coup, ou l'un après l'autre (d'abord $a$, puis $b$ sur le tableau des moyennes, les axes réduits étant gardés), donne la même **moyenne** ;
- la **variance** d'un coup, avec ddof = 0, est la moyenne, le long de $b$, des variances le long de $a$, plus la variance, le long de $b$, des moyennes le long de $a$ ;
- avec ddof = 1 partout, cette décomposition n'est plus exacte.

Écris deux fonctions, qui reçoivent la fonction à tester, `moments`, de même contrat que `moments_along_axes`. Toutes les réductions, d'un coup comme l'une après l'autre, passent par `moments` (au moins 4 appels par cas : la vérification les compte) :
- `check_two_axes(moments, x, a, b)` lève une `AssertionError` (un `assert` avec un message qui dit le cas) si les moyennes ne coïncident pas, ou si la décomposition de la variance avec ddof = 0 n'est pas exacte (`np.allclose`, `rtol=1e-9`, `atol=1e-12`) ; sinon, elle rend l'écart relatif de la même décomposition avec ddof = 1, $\frac{\lvert v_1 - (\text{moyenne des variances} + \text{variance des moyennes})\rvert}{v_1}$, le plus grand sur les valeurs des autres axes (un `float`) ;
- `property_test_17(moments)` tire 100 cas avec un générateur `np.random.default_rng(5337)`, créé dans la fonction : un tableau de 2 à 4 axes, de longueurs 2 à 5, de valeurs de loi normale, et deux axes distincts tirés au hasard ; elle rend la liste des 100 écarts de `check_two_axes`.

La cellule fournie donne trois fonctions **fausses**, construites sur la référence (`ref_tensors`) : `ignores_ddof_17` divise toujours par $N - 1$ ; `variance_of_variances_17` rend, le long de plusieurs axes, la moyenne des variances le long du premier ; `wrong_count_17` divise la somme le long de plusieurs axes par la longueur du premier seulement. La vérification lance ton test sur la référence, où il doit passer ; sur chacune des trois fonctions fausses, où il doit échouer par un `assert` ; puis sur **ta** `moments_along_axes`.

In [ ]:
def ignores_ddof_17(x, axes, ddof=0):
    """Wrong: the variance always divides by N − 1, whatever ddof (built on the reference)."""
    return ref_tensors.moments_along_axes(x, axes, ddof=1)


def variance_of_variances_17(x, axes, ddof=0):
    """Wrong: along several axes, the variance is the mean, over the other reduced axes, of the variances along the
    first one (built on the reference)."""
    axes = (int(axes),) if np.ndim(axes) == 0 else tuple(int(a) for a in axes)
    mean, var = ref_tensors.moments_along_axes(x, axes, ddof=ddof)
    if len(axes) > 1:
        var = ref_tensors.moments_along_axes(ref_tensors.moments_along_axes(x, axes[0], ddof=ddof)[1], axes[1:])[0]
    return mean, var


def wrong_count_17(x, axes, ddof=0):
    """Wrong: along several axes, the sum is divided by the length of the first axis only (built on the reference)."""
    axes = (int(axes),) if np.ndim(axes) == 0 else tuple(int(a) for a in axes)
    mean, var = ref_tensors.moments_along_axes(x, axes, ddof=ddof)
    if len(axes) > 1:
        mean = mean * np.prod([np.shape(x)[a] for a in axes]) / np.shape(x)[axes[0]]
    return mean, var


WRONG_17 = {"ignores_ddof_17": ignores_ddof_17, "variance_of_variances_17": variance_of_variances_17,
            "wrong_count_17": wrong_count_17}

In [ ]:
def check_two_axes(moments, x, a, b):
    """For the function under test moments (the contract of moments_along_axes): assert that the mean along (a, b)
    is the mean along a then b, and that the variance along (a, b) with ddof = 0 is the mean along b of the variances
    along a plus the variance along b of the means along a. Return the largest relative gap of the same decomposition
    with ddof = 1."""
    raise NotImplementedError("check_two_axes n'est pas encore écrite (04_indices.md si tu bloques)")


def property_test_17(moments):
    """check_two_axes on 100 random cases drawn with np.random.default_rng(5337): arrays of 2 to 4 axes of lengths 2
    to 5, two distinct axes. Return the list of the 100 gaps."""
    raise NotImplementedError("property_test_17 n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M20.17"):
    results_17 = []

    def report_17(ok, success, failure):
        results_17.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    calls_17 = [0]

    def counted_17(function):
        """function, its calls counted in calls_17."""
        def wrapper(x, axes, ddof=0):
            calls_17[0] += 1
            return function(x, axes, ddof=ddof)
        return wrapper

    try:
        gaps_17 = property_test_17(counted_17(ref_tensors.moments_along_axes))
    except AssertionError as exc:
        print(f"❌ Ex M20.17 : ton test échoue sur la référence, qui est juste : {exc}"[:400])
        raise StopCheck from None
    try:
        gaps_17 = np.asarray(gaps_17, dtype=float).ravel()
    except (TypeError, ValueError):
        print(f"❌ Ex M20.17 : property_test_17 doit rendre la liste des écarts, reçu {gaps_17!r}"[:200])
        raise StopCheck from None
    report_17(calls_17[0] >= 400 and gaps_17.size >= 100,
              f"sur la référence, ton test passe : {gaps_17.size} cas, {calls_17[0]} appels de moments",
              f"sur la référence, ton test ne fait que {gaps_17.size} cas et {calls_17[0]} appels de moments : au moins "
              "100 cas, et au moins quatre appels par cas")
    report_17(gaps_17.size > 0 and np.all(np.isfinite(gaps_17)) and np.all(gaps_17 >= 0) and np.min(gaps_17) > 1e-9,
              f"avec ddof = 1, la décomposition n'est jamais exacte : écart relatif de {np.min(gaps_17):.1e} à "
              f"{np.max(gaps_17):.1e}" if gaps_17.size else "",
              "les écarts rendus avec ddof = 1 doivent être des nombres positifs, non nuls : |v₁ − (moyenne des "
              "variances + variance des moyennes)| / v₁")
    for name_17, wrong_function_17 in WRONG_17.items():
        try:
            property_test_17(wrong_function_17)
            outcome_17 = "il passe"
        except AssertionError:
            outcome_17 = None
        except Exception as exc:  # noqa: BLE001 - another error than an assert
            outcome_17 = f"il lève {type(exc).__name__} : {str(exc)[:120]}"
        report_17(outcome_17 is None, f"ton test attrape {name_17} par un assert",
                  f"ton test n'attrape pas {name_17} par un assert ({outcome_17})")
    try:
        property_test_17(mymath.tensors.moments_along_axes)
        mine_17 = None
    except AssertionError as exc:
        mine_17 = str(exc)[:200]
    print("✅ ta moments_along_axes passe ton test de propriété" if mine_17 is None else
          f"❌ ta moments_along_axes échoue à ton test ({mine_17}) : si ton test passe sur la référence, l'erreur est "
          "dans ta fonction (lance aussi les tests de M20.11)")
    verdict("M20.17", all(results_17), "ton test passe sur la référence et attrape les trois fonctions fausses.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : écris la décomposition pour un tableau de forme $(n_a, n_b)$ et démontre-la (le terme croisé s'annule). Avec ddof = 1, quels dénominateurs apparaissent, et pourquoi ne se recombinent-ils pas ? Laquelle des trois fonctions fausses la propriété des moyennes attrape-t-elle, et lesquelles seule la décomposition attrape-t-elle ? Une fonction qui calcule la variance par « moyenne des carrés moins carré de la moyenne » passerait-elle ton test, et que verraient les tests de `mymath` ?

## Partie D · Défi

Fiche §5. Des probabilités proportionnelles à la taille, plafonnées à 1, et un tirage systématique : un estimateur sans biais et presque sans variance.

### Ex M20.18 — Défi : des probabilités d'inclusion proportionnelles à la taille, et un estimateur presque sans variance 🏆 ★★★ ⏱️ 25 min
**Objectif :** construire des probabilités d'inclusion proportionnelles à une taille, plafonnées à 1, et mesurer avec un tirage systématique la précision qu'elles donnent à l'estimateur de Horvitz-Thompson.  
**Prérequis :** fiche §5 ; Ex M20.12 et M20.14 · **Parcours :** C

Une population de $N = 150$ communes (cellule fournie) : leur population au dernier recensement, $x_i$ (`X_18`, connue ; quelques très grandes villes), et leur population actuelle, $y_i$, à quelques pour cent près la même (`Y_18`, inconnue de celui qui sonde ; le vrai total est `TOTAL_18`). On veut estimer le total en visitant $n = 21$ communes (`n_18`).

La cellule fournie donne aussi `systematic_sample(pi, rng)`, le **tirage systématique** : on aligne les $\pi_i$ bout à bout sur un segment de longueur $n = \sum_i\pi_i$, on tire $u$ uniforme dans $[0, 1[$ (un seul `rng.random()` par appel), et l'on prend les unités dont l'intervalle contient l'un des points $u, u + 1, \ldots, u + n - 1$. Chaque unité est tirée avec la probabilité $\pi_i$, et **exactement** $n$ unités le sont ; elle refuse (`ValueError`) un $\pi_i$ hors de $]0, 1]$ ou une somme qui n'est pas entière.

Écris `pps_probabilities(x, n)` : des probabilités proportionnelles à $x$, de somme $n$. Une unité dont la probabilité dépasserait 1 est prise d'office ($\pi_i = 1$), et le reste, $n$ moins le nombre d'unités prises d'office, est réparti à nouveau, proportionnellement à $x$, entre les autres ; on recommence jusqu'à ce qu'aucune ne dépasse 1. Un tableau de flottants ; `x` n'est pas modifié.

Puis, avec tes probabilités `pi_18` pour `X_18` et `n_18` :
- `estimates_18` : les 2 000 estimations de Horvitz-Thompson de 2 000 tirages systématiques, faits avec un générateur `np.random.default_rng(5339)` créé une fois, un appel de `systematic_sample` par tirage ;
- `equal_18` : de même, avec les probabilités égales $n/N$ et un second générateur `np.random.default_rng(5339)`.

Réponds :
- a) le nombre de communes prises d'office ($\pi_i = 1$) ;
- b) l'écart type relatif de `estimates_18`, en % (l'écart type avec ddof = 1, divisé par le vrai total ; la valeur que donne le calcul, sans l'arrondir) ;
- c) celui de `equal_18`, en % (la valeur que donne le calcul, sans l'arrondir).

La vérification essaie `pps_probabilities` sur d'autres populations et regarde la somme, l'intervalle $]0, 1]$ et les communes prises d'office ; puis elle refait la simulation avec tes probabilités : l'écart type relatif doit rester sous 2 %, celui des probabilités égales dépasser 10 %, et le biais relatif rester sous 3 erreurs types.

In [ ]:
rng_18 = np.random.default_rng(5338)
N_18, n_18 = 150, 21                                                       # 150 communes, a sample of 21
X_18 = np.round(np.exp(rng_18.normal(np.log(2500), 1.5, size=N_18)))      # the population at the last census: known
Y_18 = np.round(X_18 * (1 + rng_18.normal(0.03, 0.04, size=N_18)))        # the population today: unknown
TOTAL_18 = Y_18.sum()                                                      # the true total, to judge the estimates


def systematic_sample(pi, rng):
    """The systematic draw: the π_i laid end to end on a segment of length n = Σ π_i (an integer), u drawn uniform in
    [0, 1[ (one rng.random() per call), and the units whose interval holds one of the points u, u + 1, ..., u + n − 1.
    Each unit is drawn with the probability π_i, and exactly n units are. Returns a boolean array."""
    pi = np.asarray(pi, dtype=float)
    if pi.ndim != 1 or np.any(~(pi > 0)) or np.any(pi > 1 + 1e-12):
        raise ValueError("every π_i must be in ]0, 1]")
    n = pi.sum()
    if abs(n - round(n)) > 1e-9:
        raise ValueError(f"the sum of the π_i must be an integer, got {n!r}")
    bounds = np.concatenate([[0.0], np.cumsum(pi)])
    bounds *= round(n) / bounds[-1]                                         # exactly n at the end
    points = rng.random() + np.arange(round(n))
    units = np.minimum(np.searchsorted(bounds, points, side="right") - 1, len(pi) - 1)
    included = np.zeros(len(pi), dtype=bool)
    included[units] = True
    return included


print("les 5 plus grandes communes :", np.sort(X_18)[-5:].astype(int).tolist(), "; total au recensement :",
      int(X_18.sum()), "; n x_i / Σx > 1 pour", int(np.sum(n_18 * X_18 / X_18.sum() > 1)), "commune(s)")

In [ ]:
def pps_probabilities(x, n):
    """Inclusion probabilities proportional to x, of sum n: a unit whose probability would exceed 1 is taken with
    certainty (π = 1), and what remains of n is shared again among the others in proportion to x, until none
    exceeds 1. Returns a new array of floats."""
    raise NotImplementedError("pps_probabilities n'est pas encore écrite (04_indices.md si tu bloques)")


pi_18 = estimates_18 = equal_18 = ...
answer_M20_18a = answer_M20_18b = answer_M20_18c = ...
with wbm.attempt("M20.18"):          # pps_probabilities: ⏳ while it is not written
    pi_18 = ...           # your probabilities for X_18 and n_18
    estimates_18 = ...    # the 2 000 Horvitz-Thompson estimates, systematic draws, np.random.default_rng(5339)
    equal_18 = ...        # the same with the equal probabilities n_18 / N_18 and a second np.random.default_rng(5339)
    answer_M20_18a = ...  # a) an integer: the number of communes with π = 1
    answer_M20_18b = ...  # b) a number, as computed (not rounded): the relative standard deviation of estimates_18, in %
    answer_M20_18c = ...  # c) a number, as computed: that of equal_18, in %

In [ ]:
for letter, answer in zip("abc", [answer_M20_18a, answer_M20_18b, answer_M20_18c]):
    wbm.check(f"M20.18{letter}", answer, computed=True)

with checked("M20.18"):
    results_18 = []

    def report_18(ok, success, failure):
        results_18.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    def expected_18(x, n):
        """min(1, c x), with c such that Σ min(1, c x) = n, by bisection: another method than the statement's."""
        low, high = 0.0, 1.0 / np.min(x)
        for _ in range(200):
            middle = (low + high) / 2
            low, high = (middle, high) if np.minimum(1.0, middle * x).sum() < n else (low, middle)
        return np.minimum(1.0, high * x)

    rng_c18 = np.random.default_rng(5338)
    cases_18 = [("la population de l'énoncé", X_18, n_18)]
    for size_c18, count_c18, spread_c18 in ((30, 6, 1.6), (60, 12, 0.3), (80, 9, 2.0), (12, 11, 1.0)):
        cases_18.append((f"{size_c18} unités, n = {count_c18}",
                         np.round(np.exp(rng_c18.normal(5, spread_c18, size=size_c18))) + 1, count_c18))
    wrong_18 = []
    for label_c18, x_c18, n_c18 in cases_18:
        copy_c18 = x_c18.copy()
        got_c18 = pps_probabilities(copy_c18, n_c18)
        want_c18 = expected_18(x_c18, n_c18)
        if not np.array_equal(copy_c18, x_c18):
            wrong_18.append(f"{label_c18} : x a été modifié")
        elif got_c18 is None or np.shape(got_c18) != x_c18.shape:
            wrong_18.append(f"{label_c18} : attendu un tableau de forme {x_c18.shape}, reçu {got_c18!r}"[:160])
        elif not np.allclose(np.asarray(got_c18, dtype=float), want_c18, rtol=1e-9, atol=1e-12):
            got_c18 = np.asarray(got_c18, dtype=float)
            wrong_18.append(f"{label_c18} : ce ne sont pas les probabilités attendues (somme attendue {n_c18}, reçu "
                            f"une somme {got_c18.sum():.6g} et un plus grand π de {got_c18.max():.6g})")
    report_18(not wrong_18, f"pps_probabilities : juste sur {len(cases_18)} populations", "pps_probabilities, "
              + " ; ".join(wrong_18[:2]))
    if wrong_18:
        raise StopCheck
    pi_c18 = np.asarray(pps_probabilities(X_18, n_18), dtype=float)
    certain_c18 = pi_c18 >= 1 - 1e-12
    report_18(abs(pi_c18.sum() - n_18) < 1e-9 and np.all((pi_c18 > 0) & (pi_c18 <= 1 + 1e-12)),
              f"tes probabilités sont dans ]0, 1] et ont pour somme {n_18}", "tes probabilités doivent être dans ]0, 1] "
              f"et avoir pour somme {n_18}")
    report_18(certain_c18.any() and X_18[certain_c18].min() > X_18[~certain_c18].max(),
              "les communes prises d'office sont les plus grandes",
              "les communes prises d'office doivent être les plus grandes")
    pi_c18 = np.where(certain_c18, 1.0, pi_c18)
    equal_c18 = np.full(N_18, n_18 / N_18)
    for label_c18, probabilities_c18 in (("tes probabilités", pi_c18), ("des probabilités égales", equal_c18)):
        rng_s18 = np.random.default_rng(5339)
        estimates_c18 = np.empty(2000)
        for k_c18 in range(2000):
            drawn_c18 = systematic_sample(probabilities_c18, rng_s18)
            estimates_c18[k_c18] = np.sum(Y_18[drawn_c18] / probabilities_c18[drawn_c18])
        spread_c18 = estimates_c18.std(ddof=1) / TOTAL_18
        bias_c18 = estimates_c18.mean() / TOTAL_18 - 1
        if probabilities_c18 is pi_c18:
            report_18(spread_c18 < 0.02, f"avec {label_c18}, l'écart type relatif reste sous 2 %",
                      f"avec {label_c18}, l'écart type relatif dépasse 2 %")
            report_18(abs(bias_c18) < 3 * spread_c18 / math.sqrt(2000),
                      "le biais relatif reste sous 3 erreurs types", "le biais relatif dépasse 3 erreurs types")
        else:
            report_18(spread_c18 > 0.10, f"avec {label_c18}, l'écart type relatif dépasse 10 %",
                      f"avec {label_c18}, l'écart type relatif reste sous 10 %")
    verdict("M20.18", all(results_18), "tes probabilités rendent l'estimation de Horvitz-Thompson presque exacte.",
            "relis les lignes ❌ ci-dessus.")

**Dans ta copie** : combien de communes dépassaient 1 avant toute répartition, et combien à la fin ? Pourquoi une répartition peut-elle en faire dépasser de nouvelles ? Pourquoi l'estimation est-elle presque exacte (que vaut $y_i/\pi_i$, à quelques pour cent près, pour une commune qui n'est pas prise d'office ?), et pourquoi le tirage de Poisson de M20.14 ne pouvait-il pas faire aussi bien ? Que deviendrait la précision si une grande ville avait beaucoup changé depuis le recensement ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu suivre la forme d'un tableau à travers des réductions, des remodelages, des transpositions et la diffusion ?
2. Sais-tu écrire un calcul d'indices en notation d'Einstein, le calculer avec `einsum` et compter son coût ?
3. Sais-tu calculer moyennes et variances le long d'axes, avec le bon ddof, et standardiser le long du bon axe ?
4. Sais-tu construire un estimateur sans biais en divisant par une probabilité d'inclusion, et choisir des probabilités qui le rendent précis ?
5. Sais-tu borner le produit de $L$ facteurs $\mathbf{I} + h\mathbf{J}$ à horizon fixé ?

**La suite** : le chapitre 20 de workbookIA, qui range ses données dans des tenseurs, standardise le long d'axes choisis, divise par une probabilité et enchaîne des applications proches de l'identité ; puis l'examen blanc CP4, qui clôt la partie IV, et M21 (la convolution).